In [1]:
# imports
import os
import sys

sys.path.append(os.path.abspath(".."))  # project root

import random
import unittest
from unittest.mock import patch
from models.bot.RandomBot import RandomBot
from models.bot.BaseBot import BaseBot
from models.GameElements.Game import Game

In [2]:
# static functions
def make_state(bets, my_id=0, stack=1000, folded=()):
    """Builds a minimal server state. bets[i] is player i's current bet; my_id's stack is `stack`."""
    return {
        "players": [
            {
                "id": i,
                "bet": bet,
                "stack": stack if i == my_id else 1000,
                "folded": i in folded,
                "hole_cards": [{"rank": 14, "suit": "hearts"}, {"rank": 13, "suit": "spades"}],
            }
            for i, bet in enumerate(bets)
        ]
    }


def collect_actions(bot, state, runs=300):
    """Asks the bot for a decision many times and returns the set of actions it ever picked."""
    return {bot.choose_action(state)[0] for _ in range(runs)}

In [3]:
# Core Function Testing
class RandomBot_Core_UnitTests(unittest.TestCase):
    """
    Same layout as AllInBotTest.ipynb: setUp builds the bot, and only methods starting with 'test_' run.
    RandomBot.choose_action picks uniformly from fold, check (or call when behind) and raise (only when
    its all-in target beats the highest bet). A raise amount is always current_bet + chips.
    """

    def setUp(self):
        self.bot = RandomBot()
        self.bot.gain_chips(1000)

    def test_RandomBot_IsABaseBot(self):
        self.assertIsInstance(self.bot, BaseBot)

    # TESTING LEGAL ACTIONS
    def test_NoBetsYet_OnlyFoldCheckOrRaise(self):
        state = make_state([0, 0, 0])
        self.assertEqual(collect_actions(self.bot, state), {"fold", "check", "raise"})

    def test_FacingBet_OnlyFoldCallOrRaise(self):
        state = make_state([0, 50, 0])
        self.assertEqual(collect_actions(self.bot, state), {"fold", "call", "raise"})

    def test_StackCannotCoverBet_NeverRaises(self):
        state = make_state([0, 1500, 0])
        self.assertEqual(collect_actions(self.bot, state), {"fold", "call"})

    def test_StackExactlyMatchesBet_NeverRaises(self):
        state = make_state([0, 1000])
        self.assertEqual(collect_actions(self.bot, state), {"fold", "call"})

    def test_AlreadyAllIn_BetMatchesHighest_FoldOrCheck(self):
        self.bot.place_bet(1000)
        self.bot.lose_chips(1000)
        state = make_state([1000, 1000])
        self.assertEqual(collect_actions(self.bot, state), {"fold", "check"})

    def test_ZeroChipsAndNoBet_FoldOrCheck(self):
        self.bot.lose_chips(1000)
        state = make_state([0, 0])
        self.assertEqual(collect_actions(self.bot, state), {"fold", "check"})

    def test_OptionsOffered_AreExactlyTheLegalOnes(self):
        with patch("random.choice", side_effect=lambda options: options[0]) as mocked:
            self.bot.choose_action(make_state([0, 0]))
            self.assertEqual(mocked.call_args[0][0], ["fold", "check", "raise"])
            self.bot.choose_action(make_state([0, 50]))
            self.assertEqual(mocked.call_args[0][0], ["fold", "call", "raise"])
            self.bot.choose_action(make_state([0, 5000]))
            self.assertEqual(mocked.call_args[0][0], ["fold", "call"])

    # TESTING AMOUNTS
    def test_Raise_AmountIsBetPlusChips(self):
        self.bot.place_bet(50)
        self.bot.lose_chips(50)  # 950 behind, 50 in front
        state = make_state([50, 100, 0])
        with patch("random.choice", return_value="raise"):
            self.assertEqual(self.bot.choose_action(state), ("raise", 1000))

    def test_FoldCheckCall_AmountIsZero(self):
        for action, state in (("fold", make_state([0, 50])), ("call", make_state([0, 50])), ("check", make_state([0, 0]))):
            with patch("random.choice", return_value=action):
                self.assertEqual(self.bot.choose_action(state), (action, 0))

    def test_RaiseAmount_AlwaysBeatsHighestBet(self):
        state = make_state([0, 300, 0])
        for _ in range(300):
            action, amount = self.bot.choose_action(state)
            if action == "raise":
                self.assertGreater(amount, 300)

    # TESTING RANDOMNESS
    def test_SameSeed_GivesSameDecisions(self):
        state = make_state([0, 50, 0])
        random.seed(7)
        first = [self.bot.choose_action(state) for _ in range(20)]
        random.seed(7)
        second = [self.bot.choose_action(state) for _ in range(20)]
        self.assertEqual(first, second)

    def test_AllLegalActions_AreRoughlyUniform(self):
        state = make_state([0, 50, 0])
        random.seed(1)
        counts = {"fold": 0, "call": 0, "raise": 0}
        for _ in range(3000):
            counts[self.bot.choose_action(state)[0]] += 1
        for count in counts.values():
            self.assertAlmostEqual(count / 3000, 1 / 3, delta=0.05)

    def test_ChooseAction_DoesNotChangeChips(self):
        for _ in range(50):
            self.bot.choose_action(make_state([0, 50]))
        self.assertEqual(self.bot.get_chips(), 1000)
        self.assertEqual(self.bot.get_current_bet(), 0)

    # TESTING WITH update_from_state (the server is the source of truth)
    def test_UpdateFromState_ThenChooseAction_UsesServerStack(self):
        state = make_state([0, 20], my_id=0, stack=400)
        self.bot.update_from_state(state, 0)
        with patch("random.choice", return_value="raise"):
            self.assertEqual(self.bot.choose_action(state), ("raise", 400))

    def test_UpdateFromState_StackBelowBet_NeverRaises(self):
        state = make_state([0, 500], my_id=0, stack=300)
        self.bot.update_from_state(state, 0)
        self.assertNotIn("raise", collect_actions(self.bot, state))


unittest.main(argv=[""], exit=False, verbosity=2)

test_AllLegalActions_AreRoughlyUniform (__main__.RandomBot_Core_UnitTests.test_AllLegalActions_AreRoughlyUniform) ... ok
test_AlreadyAllIn_BetMatchesHighest_FoldOrCheck (__main__.RandomBot_Core_UnitTests.test_AlreadyAllIn_BetMatchesHighest_FoldOrCheck) ... ok
test_ChooseAction_DoesNotChangeChips (__main__.RandomBot_Core_UnitTests.test_ChooseAction_DoesNotChangeChips) ... ok
test_FacingBet_OnlyFoldCallOrRaise (__main__.RandomBot_Core_UnitTests.test_FacingBet_OnlyFoldCallOrRaise) ... ok


test_FoldCheckCall_AmountIsZero (__main__.RandomBot_Core_UnitTests.test_FoldCheckCall_AmountIsZero) ... ok
test_NoBetsYet_OnlyFoldCheckOrRaise (__main__.RandomBot_Core_UnitTests.test_NoBetsYet_OnlyFoldCheckOrRaise) ... ok
test_OptionsOffered_AreExactlyTheLegalOnes (__main__.RandomBot_Core_UnitTests.test_OptionsOffered_AreExactlyTheLegalOnes) ... ok
test_RaiseAmount_AlwaysBeatsHighestBet (__main__.RandomBot_Core_UnitTests.test_RaiseAmount_AlwaysBeatsHighestBet) ... ok
test_Raise_AmountIsBetPlusChips (__main__.RandomBot_Core_UnitTests.test_Raise_AmountIsBetPlusChips) ... ok
test_RandomBot_IsABaseBot (__main__.RandomBot_Core_UnitTests.test_RandomBot_IsABaseBot) ... ok
test_SameSeed_GivesSameDecisions (__main__.RandomBot_Core_UnitTests.test_SameSeed_GivesSameDecisions) ... ok
test_StackCannotCoverBet_NeverRaises (__main__.RandomBot_Core_UnitTests.test_StackCannotCoverBet_NeverRaises) ... ok
test_StackExactlyMatchesBet_NeverRaises (__main__.RandomBot_Core_UnitTests.test_StackExactlyMatchesB

In [4]:
# Game Function Testing
class RandomBot_Game_UnitTests(unittest.TestCase):
    """
    RandomBots seated in a real Game. Game.betting_round() is not implemented yet, so these tests apply each
    bot's choose_action result by hand the way betting_round eventually will.
    """

    def setUp(self):
        self.player_a = RandomBot()
        self.player_b = RandomBot()
        self.player_c = RandomBot()
        self.players = [self.player_a, self.player_b, self.player_c]
        for player in self.players:
            player.gain_chips(1000)

        self.game = Game(self.players)

    def state_from_game(self):
        """Server-style snapshot built from the bots' current chips and bets."""
        return {
            "players": [
                {"id": i, "bet": p.get_current_bet(), "stack": p.get_chips(), "folded": p.is_out, "hole_cards": []}
                for i, p in enumerate(self.players)
            ]
        }

    def play_action(self, bot):
        """Applies one bot's decision to the game: returns the action taken."""
        action, amount = bot.choose_action(self.state_from_game())
        if action == "raise":
            put_in = amount - bot.get_current_bet()
            bot.lose_chips(put_in)
            bot.place_bet(amount)
            self.game.pot += put_in
            self.game.set_table_bet(amount)
        elif action == "call":
            self.game.pot += bot.call(self.game.table_bet)
        elif action == "fold":
            bot.fold()
        return action

    def forced(self, bot, action):
        """Plays one decision with the random choice forced to `action`."""
        with patch("random.choice", return_value=action):
            return self.play_action(bot)

    # TESTING THE BOTS INSIDE A GAME
    def test_RandomBots_AreSeatedInGame(self):
        self.assertEqual(len(self.game.players), 3)
        for player in self.game.players:
            self.assertIsInstance(player, RandomBot)

    def test_StartHand_EachRandomBotGetsTwoCards(self):
        self.game.start_hand()
        for player in self.players:
            self.assertEqual(len(player.get_hand()), 2)
        self.assertEqual(len(self.game.community_cards), 5)

    def test_NextStage_River_BotsSeeSevenCards(self):
        for stage in ("preflop", "flop", "turn", "river"):
            self.game.next_stage(stage)
        for player in self.players:
            self.assertEqual(len(player.get_hand()), 7)

    # TESTING EACH ACTION APPLIED TO THE GAME
    def test_BotRaises_PotAndTableBetGrow(self):
        self.forced(self.player_a, "raise")
        self.assertEqual(self.player_a.get_chips(), 0)
        self.assertEqual(self.game.get_pot(), 1000)
        self.assertEqual(self.game.get_table_bet(), 1000)

    def test_BotChecks_NothingChanges(self):
        self.forced(self.player_a, "check")
        self.assertEqual(self.game.get_pot(), 0)
        self.assertEqual(self.player_a.get_chips(), 1000)

    def test_BotCallsBet_PotGrowsByTheBet(self):
        self.game.pot = 50
        self.game.table_bet = 50
        self.forced(self.player_b, "call")
        self.assertEqual(self.game.get_pot(), 100)
        self.assertEqual(self.player_b.get_chips(), 950)
        self.assertEqual(self.player_b.get_current_bet(), 50)

    def test_BotFolds_HandClearedPotUnchanged(self):
        self.game.start_hand()
        self.forced(self.player_c, "fold")
        self.assertEqual(self.player_c.get_hand(), [])
        self.assertTrue(self.player_c.is_out)
        self.assertEqual(self.game.get_pot(), 0)

    def test_RaiseThenCall_PotIsBothStacks(self):
        self.forced(self.player_a, "raise")
        self.forced(self.player_b, "call")
        self.assertEqual(self.game.get_pot(), 2000)
        self.assertEqual(self.player_b.get_chips(), 0)

    def test_ShortStackCalls_PutsInOnlyWhatItHas(self):
        self.player_b.lose_chips(700)  # b has 300
        self.forced(self.player_a, "raise")
        self.forced(self.player_b, "call")
        self.assertEqual(self.player_b.get_chips(), 0)
        self.assertEqual(self.game.get_pot(), 1300)

    def test_RoundReset_ClearsBetsKeepsPot(self):
        self.forced(self.player_a, "raise")
        self.forced(self.player_b, "call")
        for player in self.players:
            player.reset_round()
        self.game.table_bet = 0
        self.assertEqual(self.game.get_pot(), 2000)
        for player in self.players:
            self.assertEqual(player.get_current_bet(), 0)

    # TESTING REAL RANDOM PLAY
    def test_RandomPlay_ChipsAreConserved(self):
        random.seed(3)
        for stage in ("preflop", "flop", "turn", "river"):
            self.game.next_stage(stage)
            for player in self.players:
                if not player.is_out:
                    self.play_action(player)
        total = sum(p.get_chips() for p in self.players) + self.game.get_pot()
        self.assertEqual(total, 3000)

    def test_RandomPlay_ChipsNeverNegative(self):
        for seed in range(50):
            random.seed(seed)
            self.setUp()
            for _ in range(3):
                for p in self.players:
                    if not p.is_out:
                        self.play_action(p)
            for p in self.players:
                self.assertGreaterEqual(p.get_chips(), 0)

    def test_FullHand_ShowdownRunsWithoutError(self):
        random.seed(5)
        for stage in ("preflop", "flop", "turn", "river"):
            self.game.next_stage(stage)
        for player in self.players:
            self.play_action(player)
        self.game.next_stage("showdown")


unittest.main(argv=[""], exit=False, verbosity=2)

test_AllLegalActions_AreRoughlyUniform (__main__.RandomBot_Core_UnitTests.test_AllLegalActions_AreRoughlyUniform) ... ok
test_AlreadyAllIn_BetMatchesHighest_FoldOrCheck (__main__.RandomBot_Core_UnitTests.test_AlreadyAllIn_BetMatchesHighest_FoldOrCheck) ... 

ok
test_ChooseAction_DoesNotChangeChips (__main__.RandomBot_Core_UnitTests.test_ChooseAction_DoesNotChangeChips) ... ok
test_FacingBet_OnlyFoldCallOrRaise (__main__.RandomBot_Core_UnitTests.test_FacingBet_OnlyFoldCallOrRaise) ... ok
test_FoldCheckCall_AmountIsZero (__main__.RandomBot_Core_UnitTests.test_FoldCheckCall_AmountIsZero) ... ok
test_NoBetsYet_OnlyFoldCheckOrRaise (__main__.RandomBot_Core_UnitTests.test_NoBetsYet_OnlyFoldCheckOrRaise) ... ok
test_OptionsOffered_AreExactlyTheLegalOnes (__main__.RandomBot_Core_UnitTests.test_OptionsOffered_AreExactlyTheLegalOnes) ... ok
test_RaiseAmount_AlwaysBeatsHighestBet (__main__.RandomBot_Core_UnitTests.test_RaiseAmount_AlwaysBeatsHighestBet) ... ok
test_Raise_AmountIsBetPlusChips (__main__.RandomBot_Core_UnitTests.test_Raise_AmountIsBetPlusChips) ... ok
test_RandomBot_IsABaseBot (__main__.RandomBot_Core_UnitTests.test_RandomBot_IsABaseBot) ... ok
test_SameSeed_GivesSameDecisions (__main__.RandomBot_Core_UnitTests.test_SameSeed_GivesSameDe

preflop
flop: revealing three cards
turn: revealing penultimate card
river: revealing final card
showdown
preflop
flop: revealing three cards
turn: revealing penultimate card
river: revealing final card
preflop
flop: revealing three cards
turn: revealing penultimate card
river: revealing final card
